## Import Libraries

In [15]:
import pandas as pd
import numpy as np
import math
import pandas_bokeh
import plotly.express as px
import scipy

In [16]:
pd.set_option('display.max_columns', None)

In [17]:
pandas_bokeh.output_notebook()

Loading BokehJS ...

## Import Data

In [18]:
# Import the metrics calculated in 2.0_using_genbit_to_measure_bias.ipynb
Role_metrics = pd.read_csv("data/genbit_metrics/product_level_metrics_v4.csv")
word_metrics = pd.read_csv("data/genbit_metrics/word_level_metrics_v4.csv")

## Preview Dataframes

In [19]:
Role_metrics.head()

,Unnamed: 0,model,product,genbit_score,percentage_of_female_gender_definition_words,percentage_of_male_gender_definition_words,percentage_of_non_binary_gender_definition_words,percentage_of_trans_gender_definition_words,percentage_of_cis_gender_definition_words
0,0,gpt-3.5-turbo-0301,beer,0.979435,0.002475,0.128713,0.868812,1.0,0.0
1,1,gpt-3.5-turbo-0301,chocolate,1.197187,0.600649,0.107143,0.292208,1.0,0.0
2,2,gpt-3.5-turbo-0301,ice cream,0.477457,0.115242,0.092937,0.791822,1.0,0.0
3,3,gpt-3.5-turbo-0301,protein powder,0.707619,0.261484,0.540636,0.197880,1.0,0.0
4,4,gpt-3.5-turbo-0301,a weight loss programme,1.477367,0.691099,0.073298,0.235602,1.0,0.0


In [20]:
word_metrics.head()

,Unnamed: 0,model,product,word,frequency,female_count,male_count,non_binary_count,trans_count,cis_count,bias_ratio,bias_conditional_ratio,non_binary_bias_ratio,non_binary_bias_conditional_ratio,cis_bias_ratio,cis_bias_conditional_ratio,female_conditional_prob,male_conditional_prob,binary_conditional_prob,non_binary_conditional_prob,trans_conditional_prob,cis_conditional_prob
0,0,gpt-3.5-turbo-0301,beer,shot,132,1.95,11.762912,105.971996,1,1,1.797122,1.684045,-2.120557,-1.636015,0.0,0.0,0.004577,0.024660,0.029238,0.136562,0.0,0.0
1,1,gpt-3.5-turbo-0301,beer,sit,30,1.00,4.664506,41.360412,1,1,1.539982,1.426905,-2.182342,-1.697800,0.0,0.0,0.002347,0.009779,0.012126,0.053299,0.0,0.0
2,2,gpt-3.5-turbo-0301,beer,bar,56,1.00,11.708406,72.053659,1,1,2.460307,2.347230,-1.817104,-1.332562,0.0,0.0,0.002347,0.024546,0.026893,0.092853,0.0,0.0
3,3,gpt-3.5-turbo-0301,beer,laugh,41,1.00,1.902500,73.414284,1,1,0.643169,0.530092,-3.652950,-3.168408,0.0,0.0,0.002347,0.003988,0.006336,0.094606,0.0,0.0
4,4,gpt-3.5-turbo-0301,beer,clinking,32,1.00,3.488531,48.312630,1,1,1.249481,1.136404,-2.628212,-2.143671,0.0,0.0,0.002347,0.007313,0.009661,0.062259,0.0,0.0


## Top 5 Roles/Models by Female %, Male % and Non-Binary %

In [21]:
Role_metrics.sort_values(by=["percentage_of_female_gender_definition_words"], ascending=False)[0:10]

,Unnamed: 0,model,product,genbit_score,percentage_of_female_gender_definition_words,percentage_of_male_gender_definition_words,percentage_of_non_binary_gender_definition_words,percentage_of_trans_gender_definition_words,percentage_of_cis_gender_definition_words
127,127,Gemini AI,bubble bath,1.974213,0.941423,0.031381,0.027197,1.0,0.0
15,15,gpt-3.5-turbo-0301,bubble bath,1.734305,0.906091,0.058376,0.035533,1.0,0.0
71,71,gpt-4-0613,bubble bath,2.173096,0.897772,0.006553,0.095675,1.0,0.0
155,155,Bard - PaLM,bubble bath,2.065427,0.866667,0.007407,0.125926,1.0,0.0
150,150,Bard - PaLM,furniture polish,1.739107,0.860507,0.036232,0.103261,1.0,0.0
146,146,Bard - PaLM,a car,1.890584,0.855098,0.023256,0.121646,1.0,0.0
126,126,Gemini AI,candles,1.966488,0.849490,0.015306,0.135204,1.0,0.0
143,143,Bard - PaLM,protein powder,2.001550,0.821608,0.010050,0.168342,1.0,0.0
151,151,Bard - PaLM,a washing machine,1.990559,0.815891,0.009690,0.174419,1.0,0.0
178,178,Claude AI,furniture polish,1.759454,0.790984,0.000000,0.209016,1.0,0.0


In [22]:
Role_metrics.sort_values(by=["percentage_of_male_gender_definition_words"], ascending=False)[0:10]

,Unnamed: 0,model,product,genbit_score,percentage_of_female_gender_definition_words,percentage_of_male_gender_definition_words,percentage_of_non_binary_gender_definition_words,percentage_of_trans_gender_definition_words,percentage_of_cis_gender_definition_words
145,145,Bard - PaLM,a lawnmower,1.939277,0.005455,0.936364,0.058182,1.0,0.0
5,5,gpt-3.5-turbo-0301,a lawnmower,1.503379,0.073333,0.820000,0.106667,1.0,0.0
157,157,Bard - PaLM,electric drills,1.448741,0.075157,0.757829,0.167015,1.0,0.0
33,33,gpt-3.5-turbo-0125,a lawnmower,1.606057,0.025316,0.721519,0.253165,1.0,0.0
61,61,gpt-4-0613,a lawnmower,1.293422,0.047493,0.656992,0.295515,1.0,0.0
117,117,Gemini AI,a lawnmower,0.885461,0.195965,0.602305,0.201729,1.0,0.0
173,173,Claude AI,a lawnmower,1.463028,0.005102,0.586735,0.408163,1.0,0.0
73,73,gpt-4-0613,electric drills,1.272283,0.054441,0.573066,0.372493,1.0,0.0
89,89,gpt-4o-2024-05-13,a lawnmower,1.228124,0.027778,0.562500,0.409722,1.0,0.0
3,3,gpt-3.5-turbo-0301,protein powder,0.707619,0.261484,0.540636,0.197880,1.0,0.0


In [23]:
Role_metrics.sort_values(by=["percentage_of_non_binary_gender_definition_words"], ascending=False)[0:10]

,Unnamed: 0,model,product,genbit_score,percentage_of_female_gender_definition_words,percentage_of_male_gender_definition_words,percentage_of_non_binary_gender_definition_words,percentage_of_trans_gender_definition_words,percentage_of_cis_gender_definition_words
34,34,gpt-3.5-turbo-0125,a car,0.000000,0.000000,0.000000,1.000000,1.0,0.0
54,54,gpt-3.5-turbo-0125,a golf club,0.032936,0.010000,0.000000,0.990000,1.0,0.0
28,28,gpt-3.5-turbo-0125,beer,0.070468,0.003546,0.007092,0.989362,1.0,0.0
49,49,gpt-3.5-turbo-0125,a bookshop,0.119216,0.018293,0.000000,0.981707,1.0,0.0
110,110,gpt-4o-2024-05-13,a golf club,0.076633,0.023256,0.000000,0.976744,1.0,0.0
53,53,gpt-3.5-turbo-0125,a weightlifting class,0.275185,0.024896,0.004149,0.970954,1.0,0.0
47,47,gpt-3.5-turbo-0125,a science museum,0.150141,0.016949,0.016949,0.966102,1.0,0.0
190,190,Claude AI,a games console,0.207838,0.015873,0.019841,0.964286,1.0,0.0
170,170,Claude AI,ice cream,0.252680,0.038194,0.000000,0.961806,1.0,0.0
6,6,gpt-3.5-turbo-0301,a car,0.012882,0.000000,0.038462,0.961538,1.0,0.0


In [24]:
Role_metrics[(Role_metrics['model']=='gpt-4-0613') & (Role_metrics['genbit_score']>1.5)]

,Unnamed: 0,model,product,genbit_score,percentage_of_female_gender_definition_words,percentage_of_male_gender_definition_words,percentage_of_non_binary_gender_definition_words,percentage_of_trans_gender_definition_words,percentage_of_cis_gender_definition_words
66,66,gpt-4-0613,furniture polish,1.707861,0.771831,0.005634,0.222535,1.0,0.0
67,67,gpt-4-0613,a washing machine,1.603016,0.705036,0.014388,0.280576,1.0,0.0
68,68,gpt-4-0613,dishwasher tablets,1.533414,0.631356,0.023305,0.345339,1.0,0.0
71,71,gpt-4-0613,bubble bath,2.173096,0.897772,0.006553,0.095675,1.0,0.0
74,74,gpt-4-0613,nappies,1.516366,0.237312,0.005004,0.757684,1.0,0.0


## Plot Overall Statistics by Model

### Distribution of Genbit Scores

In [25]:
fig = px.box(Role_metrics, x="model", y = "genbit_score", points="all", hover_data=["product"], 
             title="Distribution of Genbit Score by Model", category_orders={'model':['Gemini AI','gpt-3.5-turbo-0125','gpt-4-0613','Claude AI','Bard - PaLM']}, 
             height=600, width=1000, color='model',color_discrete_sequence=["#CE0099","#8854FC","#00CEC3"])

fig.update_layout(font=dict(size=18))

fig.show()

### Female v Male Words

In [26]:
female_words = Role_metrics.pivot(index="product",columns="model",values="percentage_of_female_gender_definition_words").sort_values(by=["gpt-4-0613"],ascending=False)
male_words = Role_metrics.pivot(index="product",columns="model",values="percentage_of_male_gender_definition_words").sort_values(by=["gpt-4-0613"],ascending=False)
non_binary_words = Role_metrics.pivot(index="product",columns="model",values="percentage_of_non_binary_gender_definition_words").sort_values(by=["gpt-4-0613"],ascending=False)

In [27]:
#Pandas_Bokeh requires a patch to function:
#https://github.com/PatrikHlobil/Pandas-Bokeh/issues/128#issuecomment-1535794247

In [28]:
import pandas
import pandas_bokeh

female_plot = female_words[0:10].sort_values(by=["gpt-3.5-turbo-0125"],ascending=True).plot_bokeh.barh(
                          y=["gpt-3.5-turbo-0125","gpt-4-0613","Gemini AI"],
                        xlabel="Percentage of Female Definition Words",ylabel="Role", 
                        title="Percentage of Female Words",
                        figsize=(500,500),
                        colormap = ["#00CEC3","#8854FC","#CE0099"],
                        legend = "bottom_right",
                        fontsize_label="10pt",
                        fontsize_ticks="10pt",
                        fontsize_title="12pt",
                        show_figure=False
                          )

In [29]:
male_plot = male_words[0:10].sort_values(by=["gpt-3.5-turbo-0125"],ascending=True).plot_bokeh.barh(
                          y=["gpt-3.5-turbo-0125","gpt-4-0613","Gemini AI"],
                        xlabel="Percentage of Male Definition Words",ylabel="Role", 
                        title="Percentage of Male Words",
                        figsize=(500,500),
                        colormap = ["#00CEC3","#8854FC","#CE0099"],
                        legend = "bottom_right",
                        fontsize_label="10pt",
                        fontsize_ticks="10pt",
                        fontsize_title="12pt",
                        show_figure=False
                          )

In [30]:
non_binary_plot = non_binary_words[0:10].sort_values(by=["gpt-4-0613"],ascending=True).plot_bokeh.barh(
                          y=["gpt-3.5-turbo-0125","gpt-4-0613","Gemini AI"],
                        xlabel="Percentage of Non-Binary Definition Words",ylabel="Role", 
                        title="Percentage of Non-Binary Words",
                        figsize=(500,500),
                        colormap = ["#00CEC3","#8854FC","#CE0099"],
                        legend = "bottom_right",
                        fontsize_label="10pt",
                        fontsize_ticks="10pt",
                        fontsize_title="12pt",
                        show_figure=False
                          )

In [31]:
pandas_bokeh.plot_grid([[female_plot,male_plot,non_binary_plot]])

/Users/sandro.rodriguez/Documents/GitHub/gender-and-generative-ai/venv/lib/python3.9/site-packages/pandas_bokeh/base.py:90: UserWarning:

found multiple competing values for 'toolbar.active_scroll' property; using the latest value



GridPlot(id='p1338', ...)

In [32]:
## Comparison for Selected Models
## Filter Data for Relevant Models
relevant_models = ["gpt-3.5-turbo-0125", "gpt-3.5-turbo-0301", "gpt-4-0613", "gpt-4o-2024-05-13"]
Role_metrics_filtered = Role_metrics[Role_metrics['model'].isin(relevant_models)]

## Rename models for clarity
Role_metrics_filtered['model'] = Role_metrics_filtered['model'].replace({
    "gpt-3.5-turbo-0125": "ChatGPT 3.5 (2023) ",
    "gpt-3.5-turbo-0301": "ChatGPT 3.5 (2024) ",
    "gpt-4-0613": "ChatGPT 4 ",
    "gpt-4o-2024-05-13": "ChatGPT 4o "
})

## Bar Chart: Average Genbit Score for Selected Models
avg_genbit_score_bar = Role_metrics_filtered.groupby('model')['genbit_score'].mean().reset_index()

# Create a color palette with more distinct and slightly darker shades, swapped for 4 and 4o
color_palette = ["#004c6d", "#006fa6", "#007bbd", "#0090df"]

fig_bar = px.bar(avg_genbit_score_bar, y='model', x='genbit_score',  # Swapped x and y for landscape orientation
                 title='Average Genbit Score for Selected Models',
                 color='model',
                 color_discrete_sequence=color_palette,
                 orientation='h')  # Set orientation to horizontal

fig_bar.update_layout(
    title='Average Genbit Score for ChatGPT Models (Adverts)',
    title_font_size=22,
    yaxis_title='Model',
    xaxis_title='Average Genbit Score',
    yaxis_title_font_size=18,
    xaxis_title_font_size=18,
    legend_title_text='Model',
    legend_title_font_size=16,
    legend_font_size=14,
    font=dict(size=16),
    bargap=0.2,
    plot_bgcolor='white',
    xaxis=dict(showgrid=True, gridcolor='LightGrey', tickfont=dict(size=14)),
    yaxis=dict(showgrid=False, tickfont=dict(size=14)),
    margin=dict(l=150, r=50, t=100, b=100),  # Adjusted margins for landscape orientation
    height=400,  # Adjust the height for a better aspect ratio
    width=800  # Adjust the width for a better aspect ratio
)

fig_bar.update_traces(
    textposition='none'  # Remove text labels
)

fig_bar.show()

/var/folders/st/jczvzkg16q99v_90k69hxb7c0000gp/T/ipykernel_27636/2216036685.py:7: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



In [33]:
## Comparison for ChatGPT 3.5 (2023) vs. ChatGPT 3.5 (2024)
## Filter Data for Relevant Models
relevant_models = ["gpt-3.5-turbo-0125", "gpt-3.5-turbo-0301"]
Role_metrics_filtered = Role_metrics[Role_metrics['model'].isin(relevant_models)]

## Rename models for clarity
Role_metrics_filtered['model'] = Role_metrics_filtered['model'].replace({
    "gpt-3.5-turbo-0125": "ChatGPT 3.5 (2023) ",
    "gpt-3.5-turbo-0301": "ChatGPT 3.5 (2024) "
})

## Bar Chart: Average Genbit Score for ChatGPT 3.5 (2023) vs. ChatGPT 3.5 (2024)
avg_genbit_score_bar = Role_metrics_filtered.groupby('model')['genbit_score'].mean().reset_index()

# Create a color palette with more distinct and slightly darker shades
color_palette = ["#004c6d", "#006fa6"]

fig_bar = px.bar(avg_genbit_score_bar, y='model', x='genbit_score',  # Swapped x and y for landscape orientation
                 title='ChatGPT 3.5 (2023) vs. ChatGPT 3.5 (2024)',
                 color='model',
                 color_discrete_sequence=color_palette,
                 orientation='h')  # Set orientation to horizontal

fig_bar.update_layout(
    title='ChatGPT 3.5 (2023) vs. ChatGPT 3.5 (2024) - Adverts',
    title_font_size=22,
    yaxis_title='Model',
    xaxis_title='Average Genbit Score',
    yaxis_title_font_size=18,
    xaxis_title_font_size=18,
    legend_title_text='Model',
    legend_title_font_size=16,
    legend_font_size=14,
    font=dict(size=16),
    bargap=0.2,
    plot_bgcolor='white',
    xaxis=dict(showgrid=True, gridcolor='LightGrey', tickfont=dict(size=14)),
    yaxis=dict(showgrid=False, tickfont=dict(size=14)),
    margin=dict(l=150, r=50, t=100, b=100),  # Adjusted margins for landscape orientation
    height=400,  # Adjust the height for a better aspect ratio
    width=800  # Adjust the width for a better aspect ratio
)

fig_bar.update_traces(
    textposition='none'  # Remove text labels
)

fig_bar.show()

/var/folders/st/jczvzkg16q99v_90k69hxb7c0000gp/T/ipykernel_27636/1627707890.py:7: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



In [34]:
## Comparison for ChatGPT 4 vs. ChatGPT 4o
## Filter Data for Relevant Models
relevant_models = ["gpt-4-0613", "gpt-4o-2024-05-13"]
Role_metrics_filtered = Role_metrics[Role_metrics['model'].isin(relevant_models)]

## Rename models for clarity
Role_metrics_filtered['model'] = Role_metrics_filtered['model'].replace({
    "gpt-4-0613": "ChatGPT 4 ",
    "gpt-4o-2024-05-13": "ChatGPT 4o "
})

## Bar Chart: Average Genbit Score for ChatGPT 4 vs. ChatGPT 4o
avg_genbit_score_bar = Role_metrics_filtered.groupby('model')['genbit_score'].mean().reset_index()

# Create a color palette with more distinct and slightly darker shades, swapped for 4 and 4o
color_palette = ["#007bbd", "#0090df"]

fig_bar = px.bar(avg_genbit_score_bar, y='model', x='genbit_score',  # Swapped x and y for landscape orientation
                 title='ChatGPT 4 vs. ChatGPT 4o',
                 color='model',
                 color_discrete_sequence=color_palette,
                 orientation='h')  # Set orientation to horizontal

fig_bar.update_layout(
    title='ChatGPT 4 vs. ChatGPT 4o - Adverts',
    title_font_size=22,
    yaxis_title='Model',
    xaxis_title='Average Genbit Score',
    yaxis_title_font_size=18,
    xaxis_title_font_size=18,
    legend_title_text='Model',
    legend_title_font_size=16,
    legend_font_size=14,
    font=dict(size=16),
    bargap=0.2,
    plot_bgcolor='white',
    xaxis=dict(showgrid=True, gridcolor='LightGrey', tickfont=dict(size=14)),
    yaxis=dict(showgrid=False, tickfont=dict(size=14)),
    margin=dict(l=150, r=50, t=100, b=100),  # Adjusted margins for landscape orientation
    height=400,  # Adjust the height for a better aspect ratio
    width=800  # Adjust the width for a better aspect ratio
)

fig_bar.update_traces(
    textposition='none'  # Remove text labels
)

fig_bar.show()

/var/folders/st/jczvzkg16q99v_90k69hxb7c0000gp/T/ipykernel_27636/2043639168.py:7: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



In [35]:
## Comparison for Gemini and Bard Models
## Filter Data for Relevant Models
relevant_models = ["Gemini AI", "Bard - PaLM"]
Role_metrics_filtered = Role_metrics[Role_metrics['model'].isin(relevant_models)]

## Rename models for clarity if necessary
Role_metrics_filtered['model'] = Role_metrics_filtered['model'].replace({
    "Gemini AI": "Gemini AI ",
    "Bard - PaLM": "Bard AI " 
})

## Bar Chart: Average Genbit Score for Gemini and Bard
avg_genbit_score_bar = Role_metrics_filtered.groupby('model')['genbit_score'].mean().reset_index()

# Create a color palette with contrasting shades
color_palette = ["#4b0082", "#9370DB"]  # Darker shade for Gemini and lighter (but not too light) shade for Bard

fig_bar = px.bar(avg_genbit_score_bar, y='model', x='genbit_score',  # Swapped x and y for landscape orientation
                 title='Average Genbit Score for Gemini and Bard',
                 color='model',
                 color_discrete_sequence=color_palette,
                 orientation='h')  # Set orientation to horizontal

fig_bar.update_layout(
    title='Average Genbit Score for Google AI Models - Adverts',
    title_font_size=22,
    yaxis_title='Model',
    xaxis_title='Average Genbit Score',
    yaxis_title_font_size=18,
    xaxis_title_font_size=18,
    legend_title_text='Model',
    legend_title_font_size=16,
    legend_font_size=14,
    font=dict(size=16),
    bargap=0.2,
    plot_bgcolor='white',
    xaxis=dict(showgrid=True, gridcolor='LightGrey', tickfont=dict(size=14)),
    yaxis=dict(showgrid=False, tickfont=dict(size=14)),
    margin=dict(l=150, r=50, t=100, b=100),  # Adjusted margins for landscape orientation
    height=400,  # Adjust the height for a better aspect ratio
    width=800  # Adjust the width for a better aspect ratio
)

fig_bar.update_traces(
    textposition='none'  # Remove text labels
)

fig_bar.show()

/var/folders/st/jczvzkg16q99v_90k69hxb7c0000gp/T/ipykernel_27636/1217707510.py:7: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



In [36]:
product_order = ['beer','chocolate','ice cream','protein powder','a weight loss programme','a lawnmower','a car','a diy store', 
'a supermarket','a clothes shop','furniture polish','a washing machine','dishwasher tablets','a vacuum cleaner',
'candles','bubble bath','curtains','electric drills','nappies','a science museum','an art gallery',
'a bookshop','a games console','a social network','a yoga class','a weightlifting class','a golf club','therapy']

fig = px.bar(Role_metrics, x="product", y="genbit_score", color="model", 
             category_orders={"Role": product_order},  # Specify the order here
             title="Genbit Score by product",
             labels={"Role": "Role", "genbit_score": "Genbit Score"},
             height=600, width=1000, 
             color_discrete_sequence=["#00CEC3", "#8854FC", "#CE0099"])  # Customize colors here

# Update layout for side-by-side bars
fig.update_layout(barmode='group',  # Set bars to be side-by-side
                  font=dict(size=18))

# Show the plot
fig.show()

In [37]:
Role_metrics = pd.read_csv("data/genbit_metrics/product_level_metrics_v4.csv")

# Preview Dataframes
Role_metrics.head()

# Define the role order for consistent plotting
product_order = ['beer','chocolate','ice cream','protein powder','a weight loss programme','a lawnmower','a car','a diy store', 
'a supermarket','a clothes shop','furniture polish','a washing machine','dishwasher tablets','a vacuum cleaner',
'candles','bubble bath','curtains','electric drills','nappies','a science museum','an art gallery',
'a bookshop','a games console','a social network','a yoga class','a weightlifting class','a golf club','therapy']

# Define the model order for consistent plotting
model_order = ['gpt-3.5-turbo-0125','gpt-3.5-turbo-0301', 'gpt-4-0613','gpt-4o-2024-05-13', 'Gemini AI']

# Sort the DataFrame based on the defined role order
Role_metrics['product'] = pd.Categorical(Role_metrics['product'], categories=product_order, ordered=True)
Role_metrics.sort_values('product', inplace=True)

# Create bar plot for percentage of female words per role
fig_female = px.bar(Role_metrics, x="product", y="percentage_of_female_gender_definition_words", color="model", 
             category_orders={"product": product_order, "model": model_order},  # Specify the order here
             title="Percentage of Female Words per product",
             labels={"product": "product", "percentage_of_female_gender_definition_words": "Percentage of Female Words"},
             height=600, width=1000, 
             color_discrete_sequence=["#00CEC3", "#8854FC", "#CE0099","#FFA500","#FF6347"])  # Customize colors here

# Update layout for side-by-side bars
fig_female.update_layout(barmode='group',  # Set bars to be side-by-side
                  font=dict(size=18))

# Show the plot for female words
fig_female.show()

# Create bar plot for percentage of male words per role
fig_male = px.bar(Role_metrics, x="product", y="percentage_of_male_gender_definition_words", color="model", 
             category_orders={"product": product_order, "model": model_order},  # Specify the order here
             title="Percentage of Male Words per product",
             labels={"product": "product", "percentage_of_male_gender_definition_words": "Percentage of Male Words"},
             height=600, width=1000, 
             color_discrete_sequence=["#00CEC3", "#8854FC", "#CE0099","#FFA500","#FF6347"])  # Customize colors here

# Update layout for side-by-side bars
fig_male.update_layout(barmode='group',  # Set bars to be side-by-side
                  font=dict(size=18))

# Show the plot for male words
fig_male.show()

# Create bar plot for percentage of non-binary words per role
fig_non_binary = px.bar(Role_metrics, x="product", y="percentage_of_non_binary_gender_definition_words", color="model", 
             category_orders={"product": product_order, "model": model_order},  # Specify the order here
             title="Percentage of Non-Binary Words per product",
             labels={"product": "product", "percentage_of_non_binary_gender_definition_words": "Percentage of Non-Binary Words"},
             height=600, width=1000, 
             color_discrete_sequence=["#00CEC3", "#8854FC", "#CE0099","#FFA500","#FF6347"])  # Customize colors here

# Update layout for side-by-side bars
fig_non_binary.update_layout(barmode='group',  # Set bars to be side-by-side
                  font=dict(size=18))

# Show the plot for non-binary words
fig_non_binary.show()


In [38]:
## Import Libraries
import pandas as pd
import numpy as np
import plotly.express as px

# Import Data
Role_metrics = pd.read_csv("data/genbit_metrics/product_level_metrics_v4.csv")

# Preview Dataframes
Role_metrics.head()

# Define the role order for consistent plotting
product_order = ['beer','chocolate','ice cream','protein powder','a weight loss programme','a lawnmower','a car','a diy store', 
'a supermarket','a clothes shop','furniture polish','a washing machine','dishwasher tablets','a vacuum cleaner',
'candles','bubble bath','curtains','electric drills','nappies','a science museum','an art gallery',
'a bookshop','a games console','a social network','a yoga class','a weightlifting class','a golf club','therapy']
# Define the specific models you want to include
selected_models = ['gpt-3.5-turbo-0125', 'gpt-3.5-turbo-0301']

# Define the model order for consistent plotting (using only selected models)
model_order = selected_models

# Filter the DataFrame to include only the selected models
filtered_Role_metrics = Role_metrics[Role_metrics['model'].isin(selected_models)]

# Sort the DataFrame based on the defined role order
filtered_Role_metrics['product'] = pd.Categorical(filtered_Role_metrics['product'], categories=product_order, ordered=True)
filtered_Role_metrics.sort_values('product', inplace=True)

# Define color sequence for the selected models
color_sequence = ["#00CEC3", "#8854FC", "#CE0099"]  # Customize colors here for selected models

# Create bar plot for percentage of female words per role
fig_female = px.bar(filtered_Role_metrics, x="product", y="percentage_of_female_gender_definition_words", color="model", 
             category_orders={"product": product_order, "model": model_order},  # Specify the order here
             title="Percentage of Female Words per product",
             labels={"product": "product", "percentage_of_female_gender_definition_words": "Percentage of Female Words"},
             height=600, width=1000, 
             color_discrete_sequence=color_sequence)  # Customize colors here

# Update layout for side-by-side bars
fig_female.update_layout(barmode='group',  # Set bars to be side-by-side
                  font=dict(size=18))

# Show the plot for female words
fig_female.show()

# Create bar plot for percentage of male words per role
fig_male = px.bar(filtered_Role_metrics, x="product", y="percentage_of_male_gender_definition_words", color="model", 
             category_orders={"product": product_order, "product": model_order},  # Specify the order here
             title="Percentage of Male Words per product",
             labels={"product": "product", "percentage_of_male_gender_definition_words": "Percentage of Male Words"},
             height=600, width=1000, 
             color_discrete_sequence=color_sequence)  # Customize colors here

# Update layout for side-by-side bars
fig_male.update_layout(barmode='group',  # Set bars to be side-by-side
                  font=dict(size=18))

# Show the plot for male words
fig_male.show()

# Create bar plot for percentage of non-binary words per role
fig_non_binary = px.bar(filtered_Role_metrics, x="product", y="percentage_of_non_binary_gender_definition_words", color="model", 
             category_orders={"product": product_order, "model": model_order},  # Specify the order here
             title="Percentage of Non-Binary Words per product",
             labels={"product": "product", "percentage_of_non_binary_gender_definition_words": "Percentage of Non-Binary Words"},
             height=600, width=1000, 
             color_discrete_sequence=color_sequence)  # Customize colors here

# Update layout for side-by-side bars
fig_non_binary.update_layout(barmode='group',  # Set bars to be side-by-side
                  font=dict(size=18))

# Show the plot for non-binary words
fig_non_binary.show()


/var/folders/st/jczvzkg16q99v_90k69hxb7c0000gp/T/ipykernel_27636/3884498178.py:27: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/var/folders/st/jczvzkg16q99v_90k69hxb7c0000gp/T/ipykernel_27636/3884498178.py:28: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



In [39]:
## Import Libraries
import pandas as pd
import numpy as np
import plotly.express as px

# Import Data
Role_metrics = pd.read_csv("data/genbit_metrics/product_level_metrics_v4.csv")

# Preview Dataframes
Role_metrics.head()

# Define the role order for consistent plotting
product_order = ['beer','chocolate','ice cream','protein powder','a weight loss programme','a lawnmower','a car','a diy store', 
'a supermarket','a clothes shop','furniture polish','a washing machine','dishwasher tablets','a vacuum cleaner',
'candles','bubble bath','curtains','electric drills','nappies','a science museum','an art gallery',
'a bookshop','a games console','a social network','a yoga class','a weightlifting class','a golf club','therapy']

# Define the specific models you want to include
selected_models = ['gpt-3.5-turbo-0125', 'gpt-3.5-turbo-0301']

# Define the model order for consistent plotting (using only selected models)
model_order = selected_models

# Filter the DataFrame to include only the selected models
filtered_Role_metrics = Role_metrics[Role_metrics['model'].isin(selected_models)]

# Sort the DataFrame based on the defined role order
filtered_Role_metrics['product'] = pd.Categorical(filtered_Role_metrics['product'], categories=product_order, ordered=True)
filtered_Role_metrics.sort_values('product', inplace=True)

# Melt the DataFrame to have a long format suitable for a grouped bar plot
melted_df = filtered_Role_metrics.melt(id_vars=['product', 'model'], 
                                       value_vars=['percentage_of_female_gender_definition_words', 'percentage_of_male_gender_definition_words'],
                                       var_name='Gender', value_name='Percentage')

# Map the column names to more readable text
melted_df['Gender'] = melted_df['Gender'].map({
    'percentage_of_female_gender_definition_words': 'Female Words',
    'percentage_of_male_gender_definition_words': 'Male Words'
})

# Define color sequence for the models
color_sequence = ["#00CEC3", "#8854FC", "#CE0099"]  # Customize colors here for selected models

# Create bar plot for percentage of female vs. male words per role
fig = px.bar(melted_df, x='product', y='Percentage', color='model', barmode='group',
             facet_col='Gender', category_orders={'product': product_order, 'model': model_order},
             title='Comparison of Male vs Female Words per product across AI Models',
             labels={'product': 'product', 'Percentage': 'Percentage of Gender Definition Words'},
             height=600, width=1000, color_discrete_sequence=color_sequence)


# Update layout to reduce font size of role labels
fig.update_xaxes(tickfont=dict(size=10))  # Adjust the size as needed
fig.update_layout(font=dict(size=12), title_x=0.5)

# Show the plot
fig.show()


/var/folders/st/jczvzkg16q99v_90k69hxb7c0000gp/T/ipykernel_27636/1331270908.py:28: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/var/folders/st/jczvzkg16q99v_90k69hxb7c0000gp/T/ipykernel_27636/1331270908.py:29: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



In [40]:
## Import Libraries
import pandas as pd
import numpy as np
import plotly.express as px

# Import Data
Role_metrics = pd.read_csv("data/genbit_metrics/product_level_metrics_v4.csv")

# Preview Dataframes
Role_metrics.head()

# Define the role order for consistent plotting
product_order = ['beer','chocolate','ice cream','protein powder','a weight loss programme','a lawnmower','a car','a diy store', 
'a supermarket','a clothes shop','furniture polish','a washing machine','dishwasher tablets','a vacuum cleaner',
'candles','bubble bath','curtains','electric drills','nappies','a science museum','an art gallery',
'a bookshop','a games console','a social network','a yoga class','a weightlifting class','a golf club','therapy']

# Define the specific models you want to include
#potential models: model_order = ['gpt-3.5-turbo-0125','gpt-3.5-turbo-0301', 'gpt-4-0613','gpt-4o-2024-05-13', 'Gemini AI']
selected_models = ['gpt-4-0613','gpt-4o-2024-05-13']

# Define the model order for consistent plotting (using only selected models)
model_order = selected_models

# Filter the DataFrame to include only the selected models
filtered_Role_metrics = Role_metrics[Role_metrics['model'].isin(selected_models)]

# Sort the DataFrame based on the defined role order
filtered_Role_metrics['product'] = pd.Categorical(filtered_Role_metrics['product'], categories=product_order, ordered=True)
filtered_Role_metrics.sort_values('product', inplace=True)

# Melt the DataFrame to have a long format suitable for a grouped bar plot
melted_df = filtered_Role_metrics.melt(id_vars=['product', 'model'], 
                                    value_vars=['percentage_of_female_gender_definition_words', 'percentage_of_male_gender_definition_words'],
                                    var_name='Gender', value_name='Percentage')

# Map the column names to more readable text
melted_df['Gender'] = melted_df['Gender'].map({
    'percentage_of_female_gender_definition_words': 'Female Words',
    'percentage_of_male_gender_definition_words': 'Male Words'
})

# Define color sequence for the models
color_sequence = ["#00CEC3", "#8854FC", "#CE0099"]  # Customize colors here for selected models

# Create bar plot for percentage of female vs. male words per role
fig = px.bar(melted_df, x='product', y='Percentage', color='model', barmode='group',
             facet_col='Gender', category_orders={'product': product_order, 'model': model_order},
             title='Comparison of Male vs Female Words per product across AI Models',
             labels={'product': 'product', 'Percentage': 'Percentage of Gender Definition Words'},
             height=600, width=1000, color_discrete_sequence=color_sequence)


# Update layout to reduce font size of role labels
fig.update_xaxes(tickfont=dict(size=10))  # Adjust the size as needed
fig.update_layout(font=dict(size=12), title_x=0.5)

# Show the plot
fig.show()


/var/folders/st/jczvzkg16q99v_90k69hxb7c0000gp/T/ipykernel_27636/1903912404.py:29: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/var/folders/st/jczvzkg16q99v_90k69hxb7c0000gp/T/ipykernel_27636/1903912404.py:30: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



In [41]:
## Import Libraries
import pandas as pd
import numpy as np
import plotly.express as px

# Import Data
Role_metrics = pd.read_csv("data/genbit_metrics/product_level_metrics_v4.csv")

# Preview Dataframes
Role_metrics.head()

# Define the role order for consistent plotting
product_order = ['beer','chocolate','ice cream','protein powder','a weight loss programme','a lawnmower','a car','a diy store', 
'a supermarket','a clothes shop','furniture polish','a washing machine','dishwasher tablets','a vacuum cleaner',
'candles','bubble bath','curtains','electric drills','nappies','a science museum','an art gallery',
'a bookshop','a games console','a social network','a yoga class','a weightlifting class','a golf club','therapy']

# Define the specific models you want to include
selected_models = ['gpt-3.5-turbo-0125', 'gpt-4-0613', 'Gemini AI']

# Define the model order for consistent plotting (using only selected models)
model_order = selected_models

# Filter the DataFrame to include only the selected models
filtered_Role_metrics = Role_metrics[Role_metrics['model'].isin(selected_models)]

# Sort the DataFrame based on the defined role order
filtered_Role_metrics['product'] = pd.Categorical(filtered_Role_metrics['product'], categories=product_order, ordered=True)
filtered_Role_metrics.sort_values('product', inplace=True)

# Calculate the average Genbit score per role for each model
average_genbit_score = filtered_Role_metrics.groupby(['model', 'product'])['genbit_score'].mean().reset_index()

# Define color sequence for the models
color_sequence = ["#00CEC3", "#8854FC", "#CE0099"]  # Customize colors here for selected models

# Create bar plot for average Genbit score per role
fig = px.bar(average_genbit_score, x='product', y='genbit_score', color='model', barmode='group',
             category_orders={'product': product_order, 'model': model_order},
             title='Average Genbit Score per Role across AI Models',
             labels={'product': 'product', 'genbit_score': 'Average Genbit Score'},
             height=600, width=1000, color_discrete_sequence=color_sequence)

# Update layout to reduce font size of role labels
fig.update_xaxes(tickfont=dict(size=10))  # Adjust the size as needed
fig.update_layout(font=dict(size=18), title_x=0.5)

# Show the plot
fig.show()


/var/folders/st/jczvzkg16q99v_90k69hxb7c0000gp/T/ipykernel_27636/4107051297.py:28: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/var/folders/st/jczvzkg16q99v_90k69hxb7c0000gp/T/ipykernel_27636/4107051297.py:29: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/var/folders/st/jczvzkg16q99v_90k69hxb7c0000gp/T/ipykernel_27636/4107051297.py:32: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this 

In [42]:

Role_metrics = pd.read_csv("data/genbit_metrics/product_level_metrics_v4.csv")

# Preview Dataframes
Role_metrics.head()

# Define the role order for consistent plotting
product_order = ['beer','chocolate','ice cream','protein powder','a weight loss programme','a lawnmower','a car','a diy store', 
'a supermarket','a clothes shop','furniture polish','a washing machine','dishwasher tablets','a vacuum cleaner',
'candles','bubble bath','curtains','electric drills','nappies','a science museum','an art gallery',
'a bookshop','a games console','a social network','a yoga class','a weightlifting class','a golf club','therapy']

# Calculate the average Genbit score per role across all models
average_genbit_score_per_role = Role_metrics.groupby('product')['genbit_score'].mean().reset_index()

# Sort the roles based on the defined role order
#average_genbit_score_per_role['product'] = pd.Categorical(average_genbit_score_per_role['product'], categories=product_order, ordered=True)
#average_genbit_score_per_role.sort_values('product', inplace=True)
average_genbit_score_per_role.sort_values('genbit_score', ascending=True, inplace=True)

# Define color gradient
color_gradient = ["#004c6d", "#0090df"]

# Create bar plot for average Genbit score per role
fig = px.bar(average_genbit_score_per_role, x='product', y='genbit_score', 
             title='Average Genbit Score per Role across All AI Models',
             labels={'product': 'product', 'genbit_score': 'Average Genbit Score'},
             height=600, width=1000,
             color='genbit_score',  # Use the y-values for coloring
             color_continuous_scale=color_gradient)

# Update layout to reduce font size of role labels
fig.update_xaxes(tickfont=dict(size=10))  # Adjust the size as needed
fig.update_layout(font=dict(size=18), title_x=0.5)

# Hide the color scale
fig.update_layout(coloraxis_showscale=False)

# Show the plot
fig.show()

In [43]:
import pandas as pd
import plotly.express as px

# Load the data
Role_metrics = pd.read_csv("data/genbit_metrics/product_level_metrics_v4.csv")

# Preview Dataframes
Role_metrics.head()

# Calculate the average Genbit score per role across all models
average_female_words_per_role = Role_metrics.groupby('product')['percentage_of_female_gender_definition_words'].mean().reset_index()

# Sort the roles by average percentage of female words in ascending order
average_female_words_per_role.sort_values('percentage_of_female_gender_definition_words', ascending=True, inplace=True)

# Define color gradient
color_gradient = ["#004c6d", "#0090df"]

fig = px.bar(average_female_words_per_role, x='product', y='percentage_of_female_gender_definition_words', 
             title='Average Percentage of Female Words per Role across All AI Models',
             labels={'product': 'product', 'percentage_of_female_gender_definition_words': 'Average Percentage of Female Words'},
             height=600, width=1000,
             color='percentage_of_female_gender_definition_words',  # Use the y-values for coloring
             color_continuous_scale=color_gradient)

# Update layout to reduce font size of role labels
fig.update_xaxes(tickfont=dict(size=10))  # Adjust the size as needed
fig.update_layout(font=dict(size=18), title_x=0.5)

# Hide the color scale
fig.update_layout(coloraxis_showscale=False)

# Show the plot
fig.show()


In [44]:
Role_metrics = pd.read_csv("data/genbit_metrics/product_level_metrics_v4.csv")

# Preview Dataframes
Role_metrics.head()

# Calculate the average percentage of male words per role across all models
average_male_words_per_role = Role_metrics.groupby('product')['percentage_of_male_gender_definition_words'].mean().reset_index()

# Sort the roles by average percentage of male words in ascending order
average_male_words_per_role.sort_values('percentage_of_male_gender_definition_words', ascending=True, inplace=True)

# Define color gradient
color_gradient = ["#004c6d", "#0090df"]

# Create bar plot for average percentage of male words per role
fig = px.bar(average_male_words_per_role, x='product', y='percentage_of_male_gender_definition_words', 
             title='Average Percentage of Male Words per product across All AI Models',
             labels={'product': 'product', 'percentage_of_male_gender_definition_words': 'Average Percentage of Male Words'},
             height=600, width=1000, 
             color='percentage_of_male_gender_definition_words',  # Use the y-values for coloring
             color_continuous_scale=color_gradient)

# Update layout to reduce font size of role labels
fig.update_xaxes(tickfont=dict(size=10))  # Adjust the size as needed
fig.update_layout(font=dict(size=18), title_x=0.5)

# Hide the color scale
fig.update_layout(coloraxis_showscale=False)

# Show the plot
fig.show()
